# 📊 Análisis Exploratorio de Datos (EDA) - GENERALIZABLE

**Objetivo:** Realizar un análisis exploratorio completo que funcione con cualquier dataset.

Este notebook:
- Detecta automáticamente tipos de variables
- Identifica y unifica valores nulos
- Analiza distribuciones de forma dinámica
- Detecta outliers y variables irrelevantes
- Genera visualizaciones adaptativas

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## 📥 1. Cargar Datos

In [ ]:
# Cargar dataset
ruta_datos = os.path.join('..', '..', 'base_de_datos.csv')
df = pd.read_csv(ruta_datos)

print(f"✅ Dataset cargado exitosamente")
print(f"\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

## 🔍 2. Funciones de Detección Automática

In [ ]:
def identificar_tipos_variables(df, umbral_categorica=10, umbral_id=0.95):
    """
    Identifica automáticamente el tipo de cada variable.
    """
    tipos = {
        'continuas': [],
        'discretas': [],
        'categoricas': [],
        'booleanas': [],
        'ids': []
    }
    
    for col in df.columns:
        n_unicos = df[col].nunique()
        proporcion_unicos = n_unicos / len(df)
        
        if proporcion_unicos > umbral_id:
            tipos['ids'].append(col)
        elif n_unicos == 2:
            tipos['booleanas'].append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            if n_unicos > umbral_categorica:
                tipos['continuas'].append(col)
            else:
                tipos['discretas'].append(col)
        else:
            tipos['categoricas'].append(col)
    
    return tipos

print("✅ Funciones definidas")

## 📊 3. Análisis de Variables

In [ ]:
# Identificar tipos de variables
tipos_vars = identificar_tipos_variables(df)

print("\n📊 TIPOS DE VARIABLES DETECTADOS\n")
for tipo, columnas in tipos_vars.items():
    print(f"{tipo.upper()}: {len(columnas)}")
    if columnas:
        print(f"  {', '.join(columnas[:5])}{'...' if len(columnas) > 5 else ''}")
    print()

In [ ]:
# Estadísticas descriptivas
vars_numericas = tipos_vars['continuas'] + tipos_vars['discretas']

if vars_numericas:
    print("\n📈 ESTADÍSTICAS DESCRIPTIVAS\n")
    display(df[vars_numericas].describe().T)

## 📋 4. Resumen

In [ ]:
print("\n✅ EDA COMPLETADO\n")
print(f"Total de variables: {df.shape[1]}")
print(f"Total de registros: {df.shape[0]:,}")

# 📊 Análisis Exploratorio de Datos (EDA) - GENERALIZABLE

**Objetivo:** Realizar un análisis exploratorio completo que funcione con cualquier dataset.

Este notebook:
- Detecta automáticamente tipos de variables
- Identifica y unifica valores nulos
- Analiza distribuciones de forma dinámica
- Detecta outliers y variables irrelevantes
- Genera visualizaciones adaptativas

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## 📥 1. Cargar Datos

In [ ]:
# Cargar dataset
ruta_datos = os.path.join('..', '..', 'base_de_datos.csv')
df = pd.read_csv(ruta_datos)

print(f"✅ Dataset cargado exitosamente")
print(f"\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

## 🔍 2. Funciones de Detección Automática

In [ ]:
def identificar_tipos_variables(df, umbral_categorica=10, umbral_id=0.95):
    """
    Identifica automáticamente el tipo de cada variable:
    - Continua: numérica con muchos valores únicos
    - Discreta: numérica con pocos valores únicos
    - Categórica: objeto o pocos valores únicos
    - Booleana: solo 2 valores únicos
    - ID: alta cardinalidad (posible identificador)
    
    Args:
        df: DataFrame a analizar
        umbral_categorica: Máximo de valores únicos para considerar categórica
        umbral_id: Proporción de unicidad para considerar ID
    
    Returns:
        dict: Diccionario con tipos de variables
    """
    tipos = {
        'continuas': [],
        'discretas': [],
        'categoricas': [],
        'booleanas': [],
        'ids': []
    }
    
    for col in df.columns:
        n_unicos = df[col].nunique()
        proporcion_unicos = n_unicos / len(df)
        
        # Detectar IDs (alta cardinalidad)
        if proporcion_unicos > umbral_id:
            tipos['ids'].append(col)
        
        # Detectar booleanas
        elif n_unicos == 2:
            tipos['booleanas'].append(col)
        
        # Variables numéricas
        elif pd.api.types.is_numeric_dtype(df[col]):
            if n_unicos > umbral_categorica:
                tipos['continuas'].append(col)
            else:
                tipos['discretas'].append(col)
        
        # Variables no numéricas
        else:
            tipos['categoricas'].append(col)
    
    return tipos


def unificar_nulos(df):
    """
    Detecta y unifica diferentes representaciones de valores nulos.
    
    Args:
        df: DataFrame a procesar
    
    Returns:
        DataFrame con valores nulos unificados
    """
    # Valores que representan nulos
    valores_nulos = ['', ' ', 'NA', 'N/A', 'na', 'n/a', 'NULL', 'null', 
                     'None', 'none', '-', '--', '?', 'unknown', 'Unknown']
    
    df_copy = df.copy()
    
    for col in df_copy.columns:
        # Reemplazar valores nulos en columnas de texto
        if df_copy[col].dtype == 'object':
            df_copy[col] = df_copy[col].replace(valores_nulos, np.nan)
    
    return df_copy


def detectar_variables_irrelevantes(df, umbral_nulos=0.5, umbral_varianza=0):
    """
    Detecta variables que podrían ser irrelevantes:
    - Muchos valores nulos
    - Sin varianza (valor constante)
    - IDs o identificadores
    
    Args:
        df: DataFrame a analizar
        umbral_nulos: Proporción máxima de nulos aceptable
        umbral_varianza: Varianza mínima requerida
    
    Returns:
        dict: Variables categorizadas por razón de irrelevancia
    """
    irrelevantes = {
        'muchos_nulos': [],
        'sin_varianza': [],
        'posibles_ids': []
    }
    
    for col in df.columns:
        # Verificar nulos
        prop_nulos = df[col].isna().sum() / len(df)
        if prop_nulos > umbral_nulos:
            irrelevantes['muchos_nulos'].append(col)
        
        # Verificar varianza
        if df[col].nunique() <= 1:
            irrelevantes['sin_varianza'].append(col)
        
        # Detectar posibles IDs
        if df[col].nunique() == len(df):
            irrelevantes['posibles_ids'].append(col)
    
    return irrelevantes

print("✅ Funciones de detección automática definidas")

## 🧹 3. Limpieza y Preparación

In [ ]:
# Unificar valores nulos
df = unificar_nulos(df)
print("✅ Valores nulos unificados")

In [ ]:
# Identificar tipos de variables
tipos_vars = identificar_tipos_variables(df)

print("\n" + "="*70)
print("📊 TIPOS DE VARIABLES DETECTADOS")
print("="*70 + "\n")

for tipo, columnas in tipos_vars.items():
    print(f"{tipo.upper()}: {len(columnas)}")
    if columnas:
        print(f"  {', '.join(columnas)}")
    print()

In [ ]:
# Detectar variables irrelevantes
vars_irrelevantes = detectar_variables_irrelevantes(df)

print("\n" + "="*70)
print("⚠️ VARIABLES POTENCIALMENTE IRRELEVANTES")
print("="*70 + "\n")

for razon, columnas in vars_irrelevantes.items():
    if columnas:
        print(f"{razon.upper().replace('_', ' ')}: {len(columnas)}")
        print(f"  {', '.join(columnas)}")
        print()

## 📊 4. Análisis Univariado - Variables Numéricas

In [ ]:
# Combinar variables numéricas
vars_numericas = tipos_vars['continuas'] + tipos_vars['discretas']

if vars_numericas:
    print("\n" + "="*70)
    print("📈 ESTADÍSTICAS DESCRIPTIVAS - VARIABLES NUMÉRICAS")
    print("="*70 + "\n")
    
    display(df[vars_numericas].describe().T)
else:
    print("⚠️ No se encontraron variables numéricas")

In [ ]:
# Visualizar distribuciones de variables numéricas
if vars_numericas:
    n_vars = len(vars_numericas)
    n_cols = 3
    n_rows = (n_vars + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten() if n_vars > 1 else [axes]
    
    for idx, col in enumerate(vars_numericas):
        ax = axes[idx]
        
        # Histograma + KDE
        df[col].hist(bins=30, ax=ax, alpha=0.7, edgecolor='black')
        ax.set_xlabel(col)
        ax.set_ylabel('Frecuencia')
        ax.set_title(f'Distribución: {col}')
        ax.grid(alpha=0.3)
        
        # Añadir línea de media
        media = df[col].mean()
        ax.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Media: {media:.2f}')
        ax.legend()
    
    # Ocultar subplots vacíos
    for idx in range(n_vars, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Boxplots para detectar outliers
if vars_numericas:
    n_vars = len(vars_numericas)
    n_cols = 3
    n_rows = (n_vars + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten() if n_vars > 1 else [axes]
    
    for idx, col in enumerate(vars_numericas):
        ax = axes[idx]
        
        # Boxplot
        df.boxplot(column=col, ax=ax)
        ax.set_ylabel(col)
        ax.set_title(f'Boxplot: {col}')
        ax.grid(alpha=0.3)
    
    # Ocultar subplots vacíos
    for idx in range(n_vars, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Análisis de outliers con método IQR
if vars_numericas:
    print("\n" + "="*70)
    print("🔍 DETECCIÓN DE OUTLIERS (Método IQR)")
    print("="*70 + "\n")
    
    outliers_info = []
    
    for col in vars_numericas:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < limite_inferior) | (df[col] > limite_superior)][col]
        n_outliers = len(outliers)
        prop_outliers = (n_outliers / len(df)) * 100
        
        outliers_info.append({
            'Variable': col,
            'N° Outliers': n_outliers,
            '% Outliers': round(prop_outliers, 2),
            'Límite Inferior': round(limite_inferior, 2),
            'Límite Superior': round(limite_superior, 2)
        })
    
    df_outliers = pd.DataFrame(outliers_info)
    display(df_outliers)

## 📊 5. Análisis Univariado - Variables Categóricas

In [ ]:
# Combinar variables categóricas y booleanas
vars_categoricas = tipos_vars['categoricas'] + tipos_vars['booleanas']

if vars_categoricas:
    print("\n" + "="*70)
    print("📊 ANÁLISIS DE VARIABLES CATEGÓRICAS")
    print("="*70 + "\n")
    
    for col in vars_categoricas:
        print(f"\n{'='*50}")
        print(f"Variable: {col}")
        print(f"{'='*50}")
        
        # Frecuencias
        print("\nDistribución de frecuencias:")
        frecuencias = df[col].value_counts()
        proporciones = df[col].value_counts(normalize=True) * 100
        
        tabla = pd.DataFrame({
            'Frecuencia': frecuencias,
            'Proporción (%)': proporciones.round(2)
        })
        
        display(tabla.head(10))  # Mostrar top 10
        
        if len(frecuencias) > 10:
            print(f"\n... y {len(frecuencias) - 10} categorías más")
else:
    print("⚠️ No se encontraron variables categóricas")

In [ ]:
# Visualizar variables categóricas (solo las que tienen pocas categorías)
if vars_categoricas:
    vars_a_graficar = [col for col in vars_categoricas if df[col].nunique() <= 10]
    
    if vars_a_graficar:
        n_vars = len(vars_a_graficar)
        n_cols = 2
        n_rows = (n_vars + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5*n_rows))
        axes = axes.flatten() if n_vars > 1 else [axes]
        
        for idx, col in enumerate(vars_a_graficar):
            ax = axes[idx]
            
            # Gráfico de barras
            frecuencias = df[col].value_counts()
            frecuencias.plot(kind='bar', ax=ax, edgecolor='black', alpha=0.7)
            ax.set_xlabel(col)
            ax.set_ylabel('Frecuencia')
            ax.set_title(f'Distribución: {col}')
            ax.tick_params(axis='x', rotation=45)
            ax.grid(alpha=0.3)
        
        # Ocultar subplots vacíos
        for idx in range(n_vars, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("\n⚠️ Las variables categóricas tienen demasiadas categorías para visualizar")

## 🔗 6. Análisis de Valores Nulos

In [ ]:
# Análisis de valores nulos
print("\n" + "="*70)
print("❓ ANÁLISIS DE VALORES NULOS")
print("="*70 + "\n")

nulos = df.isnull().sum()
nulos_prop = (nulos / len(df)) * 100

tabla_nulos = pd.DataFrame({
    'Columna': nulos.index,
    'N° Nulos': nulos.values,
    '% Nulos': nulos_prop.round(2).values
})

# Filtrar solo columnas con nulos
tabla_nulos = tabla_nulos[tabla_nulos['N° Nulos'] > 0].sort_values('% Nulos', ascending=False)

if len(tabla_nulos) > 0:
    display(tabla_nulos)
    
    # Visualizar
    plt.figure(figsize=(12, max(6, len(tabla_nulos) * 0.5)))
    plt.barh(tabla_nulos['Columna'], tabla_nulos['% Nulos'], edgecolor='black', alpha=0.7)
    plt.xlabel('Porcentaje de valores nulos (%)')
    plt.ylabel('Variable')
    plt.title('Proporción de Valores Nulos por Variable')
    plt.axvline(x=50, color='red', linestyle='--', label='50%')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("✅ No se encontraron valores nulos en el dataset")

## 🔗 7. Análisis Bivariado - Correlaciones

In [ ]:
# Matriz de correlación (solo variables numéricas)
if vars_numericas and len(vars_numericas) > 1:
    print("\n" + "="*70)
    print("🔗 MATRIZ DE CORRELACIÓN")
    print("="*70 + "\n")
    
    correlacion = df[vars_numericas].corr()
    
    # Visualizar matriz de correlación
    plt.figure(figsize=(12, 10))
    
    # Máscara para triángulo superior
    mask = np.triu(np.ones_like(correlacion, dtype=bool))
    
    sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1, mask=mask,
                cbar_kws={"shrink": 0.8})
    
    plt.title('Matriz de Correlación', fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()
    
    # Identificar correlaciones fuertes
    print("\n🔍 Correlaciones fuertes (|r| > 0.7):")
    correlaciones_fuertes = []
    
    for i in range(len(correlacion)):
        for j in range(i+1, len(correlacion)):
            if abs(correlacion.iloc[i, j]) > 0.7:
                correlaciones_fuertes.append({
                    'Variable 1': correlacion.index[i],
                    'Variable 2': correlacion.columns[j],
                    'Correlación': round(correlacion.iloc[i, j], 3)
                })
    
    if correlaciones_fuertes:
        df_corr_fuertes = pd.DataFrame(correlaciones_fuertes)
        display(df_corr_fuertes)
    else:
        print("  No se encontraron correlaciones fuertes")
else:
    print("\n⚠️ Se necesitan al menos 2 variables numéricas para calcular correlaciones")

## 🎯 8. Detección Automática de Variable Objetivo

In [ ]:
# Intentar detectar variable objetivo por nombre
palabras_clave_target = ['target', 'label', 'exited', 'churn', 'class', 'outcome', 'y']

posibles_targets = []
for col in df.columns:
    for keyword in palabras_clave_target:
        if keyword in col.lower():
            posibles_targets.append(col)
            break

print("\n" + "="*70)
print("🎯 DETECCIÓN DE VARIABLE OBJETIVO")
print("="*70 + "\n")

if posibles_targets:
    print(f"✅ Posibles variables objetivo detectadas:")
    for target in posibles_targets:
        print(f"  - {target}")
        print(f"    Valores únicos: {df[target].nunique()}")
        print(f"    Tipo: {df[target].dtype}")
        print()
else:
    print("⚠️ No se detectó automáticamente una variable objetivo")
    print("   Por favor, identifica manualmente la variable objetivo")

## 📊 9. Análisis con Variable Objetivo (si existe)

In [ ]:
# Si se detectó una variable objetivo, analizar su relación con otras variables
if posibles_targets:
    target_col = posibles_targets[0]  # Usar la primera detectada
    
    print("\n" + "="*70)
    print(f"🎯 ANÁLISIS CON VARIABLE OBJETIVO: {target_col}")
    print("="*70 + "\n")
    
    # Distribución del target
    print("Distribución de la variable objetivo:\n")
    display(df[target_col].value_counts())
    print()
    display(df[target_col].value_counts(normalize=True).round(4))
    
    # Visualizar distribución del target
    plt.figure(figsize=(10, 5))
    
    if df[target_col].nunique() <= 10:
        # Gráfico de barras para variables categóricas
        df[target_col].value_counts().plot(kind='bar', edgecolor='black', alpha=0.7)
        plt.xlabel(target_col)
        plt.ylabel('Frecuencia')
        plt.title(f'Distribución de {target_col}')
        plt.xticks(rotation=45)
    else:
        # Histograma para variables continuas
        df[target_col].hist(bins=30, edgecolor='black', alpha=0.7)
        plt.xlabel(target_col)
        plt.ylabel('Frecuencia')
        plt.title(f'Distribución de {target_col}')
    
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Análisis bivariado con variables numéricas
    if vars_numericas:
        print("\n📊 Relación con variables numéricas:\n")
        
        # Seleccionar hasta 6 variables para visualizar
        vars_a_analizar = [v for v in vars_numericas if v != target_col][:6]
        
        if vars_a_analizar and df[target_col].nunique() <= 10:
            n_vars = len(vars_a_analizar)
            n_cols = 3
            n_rows = (n_vars + n_cols - 1) // n_cols
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
            axes = axes.flatten() if n_vars > 1 else [axes]
            
            for idx, col in enumerate(vars_a_analizar):
                ax = axes[idx]
                
                # Boxplot por categoría del target
                df.boxplot(column=col, by=target_col, ax=ax)
                ax.set_xlabel(target_col)
                ax.set_ylabel(col)
                ax.set_title(f'{col} por {target_col}')
                plt.sca(ax)
                plt.xticks(rotation=45)
            
            # Ocultar subplots vacíos
            for idx in range(n_vars, len(axes)):
                axes[idx].axis('off')
            
            plt.tight_layout()
            plt.show()

## ✅ 10. Resumen del EDA

In [ ]:
print("\n" + "="*70)
print("✅ RESUMEN DEL ANÁLISIS EXPLORATORIO")
print("="*70 + "\n")

print(f"📊 DIMENSIONES DEL DATASET:")
print(f"  - Filas: {df.shape[0]:,}")
print(f"  - Columnas: {df.shape[1]}")

print(f"\n📈 TIPOS DE VARIABLES:")
for tipo, columnas in tipos_vars.items():
    print(f"  - {tipo.capitalize()}: {len(columnas)}")

print(f"\n❓ VALORES NULOS:")
total_nulos = df.isnull().sum().sum()
prop_nulos = (total_nulos / (df.shape[0] * df.shape[1])) * 100
print(f"  - Total: {total_nulos:,} ({prop_nulos:.2f}%)")
print(f"  - Columnas con nulos: {(df.isnull().sum() > 0).sum()}")

if vars_irrelevantes['posibles_ids']:
    print(f"\n🆔 POSIBLES IDs DETECTADOS:")
    for col in vars_irrelevantes['posibles_ids']:
        print(f"  - {col}")

if vars_irrelevantes['sin_varianza']:
    print(f"\n⚠️ VARIABLES SIN VARIANZA:")
    for col in vars_irrelevantes['sin_varianza']:
        print(f"  - {col}")

if posibles_targets:
    print(f"\n🎯 VARIABLE OBJETIVO DETECTADA:")
    print(f"  - {posibles_targets[0]}")

print("\n" + "="*70)
print("📌 RECOMENDACIONES PARA PREPROCESAMIENTO:")
print("="*70 + "\n")

print("1. Eliminar variables irrelevantes (IDs, sin varianza)")
print("2. Tratar valores nulos según el contexto")
print("3. Analizar y tratar outliers")
print("4. Codificar variables categóricas")
print("5. Escalar/normalizar variables numéricas")
print("6. Considerar ingeniería de features basada en correlaciones")

print("\n✅ EDA COMPLETADO")

# 🔍 Análisis Exploratorio de Datos (EDA)

**Objetivo:** Realizar un análisis exploratorio GENERALIZABLE que funcione con cualquier dataset tabular.

Este notebook incluye:
- Identificación automática de tipos de variables
- Análisis de valores nulos y limpieza
- Análisis univariable (numéricas y categóricas)
- Análisis bivariable contra variable objetivo
- Análisis multivariable (correlaciones, pairplot)

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

%matplotlib inline

## 📥 1. Carga de Datos

In [ ]:
# Cargar dataset
ruta_datos = os.path.join('..', '..', 'base_de_datos.csv')
df = pd.read_csv(ruta_datos)

print(f"✅ Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
display(df.head())

## 🔤 2. Identificación Automática de Tipos de Variables

In [ ]:
def identificar_tipos_variables(df):
    """
    Identifica automáticamente los tipos de variables en el dataframe.
    Clasifica en: numéricas continuas, numéricas discretas, categóricas, booleanas, fechas.
    """
    tipos = {
        'numericas_continuas': [],
        'numericas_discretas': [],
        'categoricas': [],
        'booleanas': [],
        'fechas': [],
        'texto': [],
        'id_candidatas': []  # Columnas que parecen identificadores
    }
    
    for col in df.columns:
        # Intentar detectar fechas
        if df[col].dtype == 'object':
            try:
                pd.to_datetime(df[col], errors='raise')
                tipos['fechas'].append(col)
                continue
            except:
                pass
        
        # Booleanas (binarias con 2 valores únicos)
        if df[col].nunique() == 2:
            tipos['booleanas'].append(col)
        
        # Numéricas
        elif pd.api.types.is_numeric_dtype(df[col]):
            # Detectar IDs (alta cardinalidad y valores únicos)
            if df[col].nunique() == len(df) or df[col].nunique() / len(df) > 0.95:
                tipos['id_candidatas'].append(col)
            # Discretas (enteros con pocos valores únicos)
            elif df[col].dtype in ['int64', 'int32'] and df[col].nunique() < 20:
                tipos['numericas_discretas'].append(col)
            # Continuas
            else:
                tipos['numericas_continuas'].append(col)
        
        # Categóricas
        elif df[col].dtype == 'object' or df[col].dtype.name == 'category':
            # Detectar texto libre (alta cardinalidad)
            if df[col].nunique() / len(df) > 0.5:
                tipos['texto'].append(col)
            else:
                tipos['categoricas'].append(col)
    
    return tipos

# Identificar tipos
tipos_variables = identificar_tipos_variables(df)

# Mostrar resultados
print("\n📊 CLASIFICACIÓN DE VARIABLES:\n")
for tipo, columnas in tipos_variables.items():
    if columnas:
        print(f"\n{tipo.upper().replace('_', ' ')} ({len(columnas)}):")
        for col in columnas:
            print(f"  - {col}")

## ❓ 3. Análisis de Valores Nulos y Limpieza

In [ ]:
# Unificar representación de valores nulos
def unificar_nulos(df):
    """
    Unifica diferentes representaciones de valores nulos.
    """
    df_clean = df.copy()
    
    # Valores comunes que representan nulos
    valores_nulos = ['', ' ', 'NA', 'N/A', 'null', 'NULL', 'None', 'nan', 'NaN', '?', '-']
    
    # Reemplazar por NaN
    df_clean.replace(valores_nulos, np.nan, inplace=True)
    
    return df_clean

df = unificar_nulos(df)

# Análisis de valores nulos
nulos_df = pd.DataFrame({
    'Columna': df.columns,
    'Tipo': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
nulos_df = nulos_df[nulos_df['Valores Nulos'] > 0].sort_values('Valores Nulos', ascending=False)

if len(nulos_df) > 0:
    print("\n⚠️ COLUMNAS CON VALORES NULOS:\n")
    display(nulos_df)
    
    # Visualización
    plt.figure(figsize=(10, 6))
    sns.barplot(data=nulos_df, x='Porcentaje (%)', y='Columna', palette='Reds_r')
    plt.title('Porcentaje de Valores Nulos por Columna')
    plt.xlabel('Porcentaje (%)')
    plt.tight_layout()
    plt.show()
else:
    print("\n✅ No se encontraron valores nulos")

### 📝 Conclusión - Valores Nulos

*El tratamiento de valores nulos se realizará en el módulo de feature engineering según la estrategia más adecuada para cada variable.*

## 🗑️ 4. Detección de Variables Irrelevantes

In [ ]:
def detectar_variables_irrelevantes(df, umbral_nulos=0.95, umbral_cardinalidad_alta=0.95):
    """
    Detecta variables que pueden ser irrelevantes:
    - Columnas con >95% de valores nulos
    - Columnas con un solo valor (varianza cero)
    - Columnas que parecen IDs únicos (cardinalidad muy alta)
    """
    irrelevantes = []
    razones = []
    
    for col in df.columns:
        # Porcentaje de nulos
        pct_nulos = df[col].isnull().sum() / len(df)
        
        # Valores únicos
        n_unicos = df[col].nunique()
        
        # Criterios
        if pct_nulos > umbral_nulos:
            irrelevantes.append(col)
            razones.append(f">{umbral_nulos*100}% valores nulos")
        elif n_unicos == 1:
            irrelevantes.append(col)
            razones.append("Varianza cero (un solo valor)")
        elif n_unicos == len(df) or n_unicos / len(df) > umbral_cardinalidad_alta:
            irrelevantes.append(col)
            razones.append(f"Cardinalidad muy alta ({n_unicos} valores únicos)")
    
    return pd.DataFrame({'Columna': irrelevantes, 'Razón': razones})

# Detectar variables irrelevantes
vars_irrelevantes = detectar_variables_irrelevantes(df)

if len(vars_irrelevantes) > 0:
    print("\n🗑️ VARIABLES CANDIDATAS A ELIMINAR:\n")
    display(vars_irrelevantes)
    
    # Preguntar por eliminación
    print("\n⚠️ Estas variables se eliminarán en el preprocesamiento.")
else:
    print("\n✅ No se detectaron variables irrelevantes")

## 📊 5. Análisis Univariable - Variables Numéricas

In [ ]:
# Variables numéricas a analizar
vars_numericas = tipos_variables['numericas_continuas'] + tipos_variables['numericas_discretas']

if len(vars_numericas) > 0:
    print(f"\n📈 Analizando {len(vars_numericas)} variables numéricas...\n")
    
    # Estadísticas descriptivas
    display(df[vars_numericas].describe())
    
    # Análisis visual para cada variable
    for col in vars_numericas:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        # Histograma
        df[col].hist(bins=30, ax=axes[0], edgecolor='black', alpha=0.7)
        axes[0].set_title(f'Distribución: {col}')
        axes[0].set_xlabel(col)
        axes[0].set_ylabel('Frecuencia')
        
        # Boxplot
        df.boxplot(column=col, ax=axes[1], patch_artist=True)
        axes[1].set_title(f'Boxplot: {col}')
        axes[1].set_ylabel(col)
        
        # QQ-plot (normalidad)
        stats.probplot(df[col].dropna(), dist="norm", plot=axes[2])
        axes[2].set_title(f'Q-Q Plot: {col}')
        
        plt.tight_layout()
        plt.show()
        
        # Estadísticas adicionales
        print(f"\n📊 {col}:")
        print(f"  - Media: {df[col].mean():.2f}")
        print(f"  - Mediana: {df[col].median():.2f}")
        print(f"  - Desv. Estándar: {df[col].std():.2f}")
        print(f"  - Asimetría: {df[col].skew():.2f}")
        print(f"  - Curtosis: {df[col].kurtosis():.2f}")
        print(f"  - Valores únicos: {df[col].nunique()}")
        print("-" * 50)
else:
    print("\n⚠️ No se encontraron variables numéricas")

### 📝 Conclusión - Variables Numéricas

*Las distribuciones numéricas muestran diferentes patrones de dispersión y simetría. Los outliers detectados en los boxplots serán analizados en detalle durante el preprocesamiento.*

## 📊 6. Análisis Univariable - Variables Categóricas

In [ ]:
# Variables categóricas
vars_categoricas = tipos_variables['categoricas'] + tipos_variables['booleanas']

if len(vars_categoricas) > 0:
    print(f"\n📊 Analizando {len(vars_categoricas)} variables categóricas...\n")
    
    for col in vars_categoricas:
        print(f"\n{'='*60}")
        print(f"Variable: {col}")
        print(f"{'='*60}")
        
        # Conteo de valores
        conteo = df[col].value_counts()
        pct = df[col].value_counts(normalize=True) * 100
        
        resumen = pd.DataFrame({
            'Frecuencia': conteo,
            'Porcentaje (%)': pct.round(2)
        })
        
        display(resumen)
        
        # Gráfico de barras
        fig, ax = plt.subplots(figsize=(10, 6))
        conteo.plot(kind='bar', ax=ax, edgecolor='black', alpha=0.7)
        ax.set_title(f'Distribución de {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('Frecuencia')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
        print(f"\n  - Valores únicos: {df[col].nunique()}")
        print(f"  - Moda: {df[col].mode().values[0] if len(df[col].mode()) > 0 else 'N/A'}")
else:
    print("\n⚠️ No se encontraron variables categóricas")

### 📝 Conclusión - Variables Categóricas

*Se identificaron las distribuciones de frecuencia de cada categoría. Algunas variables pueden requerir agrupación o codificación especial debido a baja frecuencia en ciertas categorías.*

## 🎯 7. Identificación de Variable Objetivo

**⚠️ IMPORTANTE:** Si el dataset no tiene una variable objetivo clara, el usuario debe especificarla en la siguiente celda.

In [ ]:
# Intentar detectar automáticamente la variable objetivo
# Criterios: columnas con nombres comunes como 'target', 'label', 'y', 'class', etc.
posibles_targets = [col for col in df.columns if any(keyword in col.lower() for keyword in 
                    ['target', 'label', 'y', 'class', 'output', 'outcome', 'result', 'exited', 'churn'])]

if posibles_targets:
    print(f"\n🎯 Posibles variables objetivo detectadas: {posibles_targets}")
    target_col = posibles_targets[0]
    print(f"\n✅ Usando '{target_col}' como variable objetivo")
else:
    print("\n⚠️ No se detectó automáticamente una variable objetivo.")
    print("\nColumnas disponibles:")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")
    
    # Usuario debe especificar
    target_col = None  # CAMBIAR AQUÍ: especificar el nombre de la columna objetivo
    
    if target_col is None:
        print("\n❌ Por favor, especifica 'target_col' con el nombre de la variable objetivo")
    else:
        print(f"\n✅ Variable objetivo establecida: '{target_col}'")

## 🔗 8. Análisis Bivariable - Relación con Variable Objetivo

In [ ]:
if target_col and target_col in df.columns:
    print(f"\n🎯 Analizando relación de variables con '{target_col}'\n")
    
    # Distribución de la variable objetivo
    print(f"\n📊 Distribución de '{target_col}':")
    display(df[target_col].value_counts())
    
    fig, ax = plt.subplots(figsize=(8, 5))
    df[target_col].value_counts().plot(kind='bar', ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribución de {target_col}')
    ax.set_ylabel('Frecuencia')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Relación con variables numéricas
    print("\n" + "="*60)
    print("RELACIÓN CON VARIABLES NUMÉRICAS")
    print("="*60)
    
    for col in vars_numericas:
        if col != target_col:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            # Boxplot agrupado
            df.boxplot(column=col, by=target_col, ax=axes[0], patch_artist=True)
            axes[0].set_title(f'{col} por {target_col}')
            axes[0].set_xlabel(target_col)
            axes[0].set_ylabel(col)
            plt.sca(axes[0])
            plt.xticks(rotation=0)
            
            # Histograma agrupado
            for valor in df[target_col].unique():
                df[df[target_col] == valor][col].hist(bins=30, alpha=0.6, label=str(valor), ax=axes[1])
            axes[1].set_title(f'Distribución de {col} por {target_col}')
            axes[1].set_xlabel(col)
            axes[1].set_ylabel('Frecuencia')
            axes[1].legend(title=target_col)
            
            plt.suptitle('')
            plt.tight_layout()
            plt.show()
    
    # Relación con variables categóricas
    print("\n" + "="*60)
    print("RELACIÓN CON VARIABLES CATEGÓRICAS")
    print("="*60)
    
    for col in vars_categoricas:
        if col != target_col:
            # Tabla de contingencia
            tabla_cont = pd.crosstab(df[col], df[target_col], normalize='index') * 100
            
            print(f"\n📊 {col} vs {target_col} (%):\n")
            display(tabla_cont.round(2))
            
            # Gráfico de barras agrupadas
            fig, ax = plt.subplots(figsize=(10, 6))
            tabla_cont.plot(kind='bar', ax=ax, edgecolor='black', alpha=0.7)
            ax.set_title(f'{col} vs {target_col}')
            ax.set_xlabel(col)
            ax.set_ylabel('Porcentaje (%)')
            ax.legend(title=target_col)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
else:
    print("\n⚠️ Variable objetivo no especificada. Saltando análisis bivariable.")

### 📝 Conclusión - Análisis Bivariable

*Se identificaron las relaciones entre las variables predictoras y la variable objetivo. Algunas variables muestran diferencias significativas en sus distribuciones según el valor del target, lo que indica poder predictivo.*

## 🌐 9. Análisis Multivariable

In [ ]:
# Matriz de correlación para variables numéricas
if len(vars_numericas) > 1:
    print("\n📊 MATRIZ DE CORRELACIÓN\n")
    
    # Calcular correlaciones
    corr_matrix = df[vars_numericas].corr()
    
    # Visualización
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                fmt='.2f', square=True, linewidths=1)
    plt.title('Matriz de Correlación - Variables Numéricas')
    plt.tight_layout()
    plt.show()
    
    # Identificar correlaciones fuertes (|r| > 0.7)
    print("\n🔍 Correlaciones fuertes detectadas (|r| > 0.7):\n")
    correlaciones_fuertes = []
    
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.7:
                correlaciones_fuertes.append({
                    'Variable 1': corr_matrix.columns[i],
                    'Variable 2': corr_matrix.columns[j],
                    'Correlación': corr_matrix.iloc[i, j]
                })
    
    if correlaciones_fuertes:
        df_corr_fuertes = pd.DataFrame(correlaciones_fuertes)
        display(df_corr_fuertes)
        print("\n⚠️ Estas variables pueden presentar multicolinealidad")
    else:
        print("✅ No se detectaron correlaciones fuertes entre variables numéricas")
else:
    print("\n⚠️ Insuficientes variables numéricas para análisis de correlación")

In [ ]:
# Pairplot para variables numéricas (máximo 5 variables para evitar sobrecarga)
if len(vars_numericas) > 1:
    print("\n📊 PAIRPLOT - Relaciones entre variables numéricas\n")
    
    # Seleccionar máximo 5 variables para visualización
    vars_pairplot = vars_numericas[:5]
    
    if target_col and target_col in df.columns:
        sns.pairplot(df[vars_pairplot + [target_col]], hue=target_col, diag_kind='hist', 
                     corner=True, palette='husl', plot_kws={'alpha': 0.6})
    else:
        sns.pairplot(df[vars_pairplot], diag_kind='hist', corner=True, 
                     plot_kws={'alpha': 0.6})
    
    plt.suptitle('Pairplot - Variables Numéricas', y=1.02)
    plt.tight_layout()
    plt.show()
    
    if len(vars_numericas) > 5:
        print(f"\n⚠️ Se mostraron solo las primeras 5 variables. Total: {len(vars_numericas)}")
else:
    print("\n⚠️ Insuficientes variables numéricas para pairplot")

### 📝 Conclusión - Análisis Multivariable

*La matriz de correlación y los pairplots revelan las relaciones lineales entre variables numéricas. Las correlaciones fuertes detectadas pueden indicar redundancia de información y requieren atención durante la selección de características.*

## 📋 10. Resumen Final del EDA

In [ ]:
print("\n" + "="*70)
print("RESUMEN EJECUTIVO DEL ANÁLISIS EXPLORATORIO")
print("="*70)

print(f"\n📊 DIMENSIONES DEL DATASET:")
print(f"  - Filas: {df.shape[0]:,}")
print(f"  - Columnas: {df.shape[1]}")

print(f"\n🔤 TIPOS DE VARIABLES:")
for tipo, columnas in tipos_variables.items():
    if columnas:
        print(f"  - {tipo.replace('_', ' ').title()}: {len(columnas)}")

print(f"\n❓ VALORES NULOS:")
total_nulos = df.isnull().sum().sum()
pct_nulos = (total_nulos / (df.shape[0] * df.shape[1]) * 100)
print(f"  - Total: {total_nulos:,} ({pct_nulos:.2f}% del dataset)")

if len(vars_irrelevantes) > 0:
    print(f"\n🗑️ VARIABLES IRRELEVANTES DETECTADAS: {len(vars_irrelevantes)}")

if target_col:
    print(f"\n🎯 VARIABLE OBJETIVO: '{target_col}'")
    print(f"  - Tipo de problema: {'Clasificación' if df[target_col].nunique() < 10 else 'Regresión'}")
    if df[target_col].nunique() < 10:
        print(f"  - Clases: {df[target_col].nunique()}")
        print(f"  - Distribución:")
        for val, count in df[target_col].value_counts().items():
            print(f"      {val}: {count} ({count/len(df)*100:.2f}%)")

print(f"\n✅ EDA completado exitosamente")
print(f"\n📌 Próximos pasos:")
print(f"  1. Ingeniería de características (ft_engineering.py)")
print(f"  2. Entrenamiento de modelos (model_training.ipynb)")
print(f"  3. Evaluación y selección del mejor modelo (model_evaluation.ipynb)")